In [ ]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [ ]:
!pip install -q jupysql

%reload_ext sql
%config SqlMagic.displaylimit = 50
%sql engine

In [ ]:
from ucimlrepo import fetch_ucirepo

# id=2 Adult; id=222 Bank Marketing; id=560 Seoul Bike Sharing Demand
dataset = fetch_ucirepo(id=2)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

csv_path = '/content/hw3_dataset.csv'
df.to_csv(csv_path, index=False)

print(df.shape)
print(df.head())

(48842, 15)
   age         workclass  fnlwgt  education  education-num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   
3   53           Private  234721       11th              7   
4   28           Private  338409  Bachelors             13   

       marital-status         occupation   relationship   race     sex  \
0       Never-married       Adm-clerical  Not-in-family  White    Male   
1  Married-civ-spouse    Exec-managerial        Husband  White    Male   
2            Divorced  Handlers-cleaners  Not-in-family  White    Male   
3  Married-civ-spouse  Handlers-cleaners        Husband  Black    Male   
4  Married-civ-spouse     Prof-specialty           Wife  Black  Female   

   capital-gain  capital-loss  hours-per-week native-country income  
0          2174             0              40  United-States  <=50K  
1             0             0 

In [ ]:
%%sql
DROP TABLE IF EXISTS hw3_data_clean CASCADE;
DROP TABLE IF EXISTS hw3_data_staging CASCADE;

-- UCI Adult (id=2): 14 features + income target, all loaded as raw text
CREATE TABLE hw3_data_staging (
    age_raw             TEXT,  -- age
    workclass_raw       TEXT,  -- workclass
    fnlwgt_raw          TEXT,  -- fnlwgt
    education_raw       TEXT,  -- education
    education_num_raw   TEXT,  -- education-num
    marital_status_raw  TEXT,  -- marital-status
    occupation_raw      TEXT,  -- occupation
    relationship_raw    TEXT,  -- relationship
    race_raw            TEXT,  -- race
    sex_raw             TEXT,  -- sex
    capital_gain_raw    TEXT,  -- capital-gain
    capital_loss_raw    TEXT,  -- capital-loss
    hours_per_week_raw  TEXT,  -- hours-per-week
    native_country_raw  TEXT,  -- native-country
    income_raw          TEXT   -- income (target)
);

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

++
||
++
++

In [ ]:
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(
                """
                COPY hw3_data_staging (
                    age_raw,
                    workclass_raw,
                    fnlwgt_raw,
                    education_raw,
                    education_num_raw,
                    marital_status_raw,
                    occupation_raw,
                    relationship_raw,
                    race_raw,
                    sex_raw,
                    capital_gain_raw,
                    capital_loss_raw,
                    hours_per_week_raw,
                    native_country_raw,
                    income_raw
                )
                FROM STDIN
                WITH (
                    FORMAT csv,
                    HEADER true,
                    DELIMITER ','
                )
                """,
                file
            )
    raw_conn.commit()
finally:
    raw_conn.close()

pd.read_sql('SELECT COUNT(*) AS loaded_rows FROM hw3_data_staging', engine)

,loaded_rows
0,48842


### Фінальна типізована таблиця `hw3_data_clean`

#### Колонки

- **`person_id`**
  - *Семантика:* сурогатний ключ запису (у датасеті немає природного ідентифікатора).
  - *Source формат:* відсутній у джерелі.
  - *Тип:* `BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY`.
  - *Очищення:* генерується автоматично.

- **`age`**
  - *Семантика:* вік особи, повних років.
  - *Source формат:* ціле число як текст, `17`–`90`.
  - *Тип:* `SMALLINT NOT NULL`, `CHECK (age BETWEEN 0 AND 120)` — вік є малим цілим, 2 байти достатньо; `CHECK` відсікає фізично неможливі значення.
  - *Очищення:* `TRIM`, `NULLIF('')`, `::SMALLINT`.

- **`education_num`**
  - *Семантика:* порядковий код рівня освіти (1 = Preschool … 16 = Doctorate). Це впорядкований код, а не кількість років навчання; він один-до-одного відповідає `education` (16 пар).
  - *Source формат:* ціле число як текст, `1`–`16`.
  - *Тип:* `SMALLINT NOT NULL`, `CHECK (education_num BETWEEN 1 AND 16)`.
  - *Очищення:* `TRIM`, `NULLIF('')`, `::SMALLINT`.

- **`hours_per_week`**
  - *Семантика:* робочі години на тиждень.
  - *Source формат:* ціле число як текст, `1`–`99`.
  - *Тип:* `SMALLINT NOT NULL`, `CHECK (hours_per_week BETWEEN 1 AND 168)` — 168 є кількістю годин у тижні.
  - *Очищення:* `TRIM`, `NULLIF('')`, `::SMALLINT`. Значення 99 — верхня межа кодування в переписі, залишаємо як є.

- **`fnlwgt`**
  - *Семантика:* вага вибірки перепису — скільки людей у популяції представляє запис.
  - *Source формат:* ціле число як текст, `12 285`–`1 490 400`.
  - *Тип:* `INTEGER NOT NULL`, `CHECK (fnlwgt > 0)` — значення перевищують межу `SMALLINT` (32 767), тому потрібен 4-байтний `INTEGER`; вага не може бути нульовою чи від'ємною.
  - *Очищення:* `TRIM`, `NULLIF('')`, `::INTEGER`.

- **`capital_gain`, `capital_loss`**
  - *Семантика:* прибуток / збиток від операцій з капіталом за рік, цілі долари США.
  - *Source формат:* ціле число як текст, `0`–`99 999` і `0`–`4 356`.
  - *Тип:* `INTEGER NOT NULL`, `CHECK (... >= 0)` — суми вже округлені до цілих доларів, тож `NUMERIC(10,2)` не дав би додаткової точності, а `INTEGER` дешевший для агрегацій. Збиток зберігається як додатне число в окремій колонці, тому обидві `>= 0`.
  - *Очищення:* `TRIM`, `NULLIF('')`, `::INTEGER`. `99999` — верхня межа кодування в переписі, не помилка.

- **`workclass`, `occupation`, `native_country`**
  - *Семантика:* тип зайнятості (8 категорій), професія (14), країна походження (41).
  - *Source формат:* текстові мітки через дефіс (`Self-emp-not-inc`); пропуски — `NULL` або `'?'`.
  - *Тип:* `TEXT NOT NULL` — у PostgreSQL `TEXT` і `VARCHAR(n)` зберігаються однаково, а штучна межа довжини нічого не перевіряє. Рядки з пропусками відсікаються, тому `NOT NULL` гарантує, що у фінальній таблиці вони не з'являться.
  - *Очищення:* `TRIM`, `NULLIF('')`, `NULLIF('?')` — обидва види пропусків зводимо до єдиного SQL `NULL`, а потім такі рядки відсікаємо в `WHERE ... IS NOT NULL`. Написання міток (`Holand-Netherlands`, `Trinadad&Tobago`) не виправляємо — це оригінальні коди датасету.

- **`education`, `marital_status`, `relationship`, `race`**
  - *Семантика:* рівень освіти (16 категорій), сімейний стан (7), роль у домогосподарстві (6), раса (5).
  - *Source формат:* текстові мітки, без пропусків.
  - *Тип:* `TEXT NOT NULL` — пропусків немає, тому `NOT NULL` фіксує це як інваріант.
  - *Очищення:* `TRIM`, `NULLIF('')`.

- **`sex`**
  - *Семантика:* стать.
  - *Source формат:* `Male` / `Female`.
  - *Тип:* `TEXT NOT NULL`, `CHECK (sex IN ('Male', 'Female'))` — домен із двох значень закріплюємо `CHECK`.
  - *Очищення:* `TRIM`, `NULLIF('')`.

- **`income_over_50k`**
  - *Семантика:* цільова мітка — річний дохід понад 50 000 $.
  - *Source формат:* `<=50K`, `>50K`, `<=50K.`, `>50K.`.
  - *Тип:* `BOOLEAN NOT NULL` — бінарна мітка природно лягає в `BOOLEAN`: 4 текстові варіанти зводяться до 2 значень, і частку зручно рахувати через `AVG(income_over_50k::INT)`.
  - *Очищення:* `TRIM`, прибираємо крапку `RTRIM(..., '.')`, потім `CASE`: `'>50K'` → `TRUE`, `'<=50K'` → `FALSE`, інше → `NULL` (відсікається в `WHERE`).

- **`from_test_split`**
  - *Семантика:* чи походить запис з оригінального тестового файлу `adult.test`.
  - *Source формат:* крапка в кінці `income_raw`.
  - *Тип:* `BOOLEAN NOT NULL`.
  - *Очищення:* `TRIM(income_raw) LIKE '%.'`. Крапка — єдиний слід оригінального поділу train/test, тому зберігаємо його окремою колонкою, щоб нормалізація мітки не знищила цю інформацію.

#### Відсікання рядків

З 48 842 рядків staging до `hw3_data_clean` потрапляє 45 175; відсічено 3 667:

- **Рядки з пропусками — 3 620.** Рядок відкидається, якщо хоча б в одній із колонок `workclass`, `occupation`, `native_country` після очищення стоїть `NULL`. Так само чинить оригінальна документація UCI Adult; ціною цього є втрата ≈ 7,4 % даних, зате всі колонки фінальної таблиці `NOT NULL`.
- **Точні дублікати — 47.** Дублікатом вважається рядок, у якого збігаються всі 15 полів датасету. Залишаємо одну копію через `SELECT DISTINCT ON (...)`; `ORDER BY ..., from_test_split` гарантує, що якщо дублікат є і в train, і в test, залишиться train-копія (`FALSE` сортується перед `TRUE`).

Перевірка після `INSERT` підтверджує, що в staging рядків більше, ніж у clean.

In [ ]:
%%sql
DROP TABLE IF EXISTS hw3_data_clean CASCADE;

CREATE TABLE hw3_data_clean (
    person_id        BIGINT
        GENERATED BY DEFAULT AS IDENTITY
        PRIMARY KEY,

    age              SMALLINT NOT NULL
        CHECK (age BETWEEN 0 AND 120),

    workclass        TEXT NOT NULL,

    fnlwgt           INTEGER NOT NULL
        CHECK (fnlwgt > 0),

    education        TEXT NOT NULL,
    education_num    SMALLINT NOT NULL
        CHECK (education_num BETWEEN 1 AND 16),

    marital_status   TEXT NOT NULL,
    occupation       TEXT NOT NULL,
    relationship     TEXT NOT NULL,
    race             TEXT NOT NULL,

    sex              TEXT NOT NULL
        CHECK (sex IN ('Male', 'Female')),

    capital_gain     INTEGER NOT NULL
        CHECK (capital_gain >= 0),
    capital_loss     INTEGER NOT NULL
        CHECK (capital_loss >= 0),

    hours_per_week   SMALLINT NOT NULL
        CHECK (hours_per_week BETWEEN 1 AND 168),

    native_country   TEXT NOT NULL,

    income_over_50k  BOOLEAN NOT NULL,
    from_test_split  BOOLEAN NOT NULL
);

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

++
||
++
++

In [ ]:
%%sql
WITH parsed AS (
    SELECT
        NULLIF(TRIM(age_raw), '')::SMALLINT            AS age,

        NULLIF(NULLIF(TRIM(workclass_raw), ''), '?')   AS workclass,

        NULLIF(TRIM(fnlwgt_raw), '')::INTEGER          AS fnlwgt,

        NULLIF(TRIM(education_raw), '')                AS education,
        NULLIF(TRIM(education_num_raw), '')::SMALLINT  AS education_num,

        NULLIF(TRIM(marital_status_raw), '')           AS marital_status,
        NULLIF(NULLIF(TRIM(occupation_raw), ''), '?')  AS occupation,
        NULLIF(TRIM(relationship_raw), '')             AS relationship,
        NULLIF(TRIM(race_raw), '')                     AS race,
        NULLIF(TRIM(sex_raw), '')                      AS sex,

        NULLIF(TRIM(capital_gain_raw), '')::INTEGER    AS capital_gain,
        NULLIF(TRIM(capital_loss_raw), '')::INTEGER    AS capital_loss,

        NULLIF(TRIM(hours_per_week_raw), '')::SMALLINT AS hours_per_week,

        NULLIF(NULLIF(TRIM(native_country_raw), ''), '?') AS native_country,

        CASE RTRIM(TRIM(income_raw), '.')
            WHEN '>50K'  THEN TRUE
            WHEN '<=50K' THEN FALSE
        END                                            AS income_over_50k,

        TRIM(income_raw) LIKE '%.'                     AS from_test_split
    FROM hw3_data_staging
)
INSERT INTO hw3_data_clean (
    age,
    workclass,
    fnlwgt,
    education,
    education_num,
    marital_status,
    occupation,
    relationship,
    race,
    sex,
    capital_gain,
    capital_loss,
    hours_per_week,
    native_country,
    income_over_50k,
    from_test_split
)
-- Дедуплікація: один рядок на кожну унікальну комбінацію 15 полів датасету.
-- ORDER BY ... from_test_split залишає train-копію (FALSE < TRUE), якщо дублікат є в обох частинах.
SELECT DISTINCT ON (
    age, workclass, fnlwgt, education, education_num, marital_status,
    occupation, relationship, race, sex, capital_gain, capital_loss,
    hours_per_week, native_country, income_over_50k
)
    age,
    workclass,
    fnlwgt,
    education,
    education_num,
    marital_status,
    occupation,
    relationship,
    race,
    sex,
    capital_gain,
    capital_loss,
    hours_per_week,
    native_country,
    income_over_50k,
    from_test_split
FROM parsed
WHERE age BETWEEN 0 AND 120
  AND workclass IS NOT NULL
  AND fnlwgt > 0
  AND education IS NOT NULL
  AND education_num BETWEEN 1 AND 16
  AND marital_status IS NOT NULL
  AND occupation IS NOT NULL
  AND relationship IS NOT NULL
  AND race IS NOT NULL
  AND sex IN ('Male', 'Female')
  AND capital_gain >= 0
  AND capital_loss >= 0
  AND hours_per_week BETWEEN 1 AND 168
  AND native_country IS NOT NULL
  AND income_over_50k IS NOT NULL
  AND from_test_split IS NOT NULL
ORDER BY
    age, workclass, fnlwgt, education, education_num, marital_status,
    occupation, relationship, race, sex, capital_gain, capital_loss,
    hours_per_week, native_country, income_over_50k,
    from_test_split;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

45175 rows affected.

++
||
++
++

# Завдання 3. Data-quality audit

## 3.1. Контроль кількості рядків

In [ ]:
%%sql
SELECT 'staging' AS layer, COUNT(*) AS n_rows
FROM hw3_data_staging

UNION ALL

SELECT 'clean' AS layer, COUNT(*) AS n_rows
FROM hw3_data_clean;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

2 rows affected.

layer,n_rows
staging,48842
clean,45175


**Результат:** у staging 48 842 рядки, у clean — 45 175. Кількість відрізняється на **3 667** рядків (≈ 7,5 %), бо під час `INSERT INTO hw3_data_clean ... SELECT` спрацьовують два фільтри.

**Правило 1. Рядки з пропусками — відсічено 3 620.**

Рядок відкидається, якщо хоча б в одній із колонок `workclass`, `occupation`, `native_country` сире значення є `NULL`, порожнім рядком або `'?'`. Точне правило в SQL:

```sql
-- у CTE parsed:
NULLIF(NULLIF(TRIM(workclass_raw), ''), '?') AS workclass   -- аналогічно occupation, native_country
-- у WHERE:
AND workclass IS NOT NULL
AND occupation IS NOT NULL
AND native_country IS NOT NULL
```

У сирих даних UCI Adult пропуск позначається `'?'`, а порожні поля CSV `COPY` перетворив на `NULL`; обидва варіанти означають відсутнє значення. Інші 12 колонок пропусків не мають.

**Правило 2. Точні дублікати — відсічено 47.**

Серед рядків, що пройшли правило 1, залишається один рядок на кожну унікальну комбінацію всіх 15 полів датасету:

```sql
SELECT DISTINCT ON (age, workclass, fnlwgt, education, education_num, marital_status,
                    occupation, relationship, race, sex, capital_gain, capital_loss,
                    hours_per_week, native_country, income_over_50k) ...
ORDER BY <ті самі 15 полів>, from_test_split
```

`from_test_split` не входить у ключ дедуплікації, бо це не властивість особи, а походження запису. Якщо однаковий рядок є і в train, і в test, сортування за ним (`FALSE` < `TRUE`) залишає train-копію.

**Інші умови `WHERE` не відсікли жодного рядка.** Перевірки діапазонів (`age BETWEEN 0 AND 120`, `hours_per_week BETWEEN 1 AND 168`, `education_num BETWEEN 1 AND 16`, `fnlwgt > 0`, `capital_gain/loss >= 0`), домену `sex` і розпізнавання мітки `income` проходять усі рядки: сирі дані вже в межах (`age` 17–90, `hours_per_week` 1–99, `income` — лише 4 відомі варіанти). Ці умови дублюють `CHECK`-обмеження таблиці й залишені як захист: якщо в джерелі з'явиться некоректне значення, рядок буде відфільтровано, а `INSERT` не впаде на порушенні `CHECK`.

**Баланс:** 48 842 − 3 620 − 47 = 45 175 ✔

## 3.2. Перевірка дублікатів

**Вибір business key.** У переписі немає ідентифікатора респондента (дані анонімізовані), тому запис про людину можна ідентифікувати лише її повним профілем. Business key — **усі 14 ознак особи**:

`age, workclass, fnlwgt, education, education_num, marital_status, occupation, relationship, race, sex, capital_gain, capital_loss, hours_per_week, native_country`

У ключ **не входять**:
- `person_id` — технічний surrogate key;
- `income_over_50k` — це цільова мітка, факт *про* особу, а не частина її ідентичності. Два записи з однаковим профілем, але різною міткою — це дублікат із суперечливими даними, і саме такі випадки перевірка має знаходити;
- `from_test_split` — походження запису (який файл), а не властивість людини.


In [ ]:
%%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (
        age, workclass, fnlwgt, education, education_num,
        marital_status, occupation, relationship, race, sex,
        capital_gain, capital_loss, hours_per_week, native_country
    )) AS distinct_business_keys,
    COUNT(*) - COUNT(DISTINCT (
        age, workclass, fnlwgt, education, education_num,
        marital_status, occupation, relationship, race, sex,
        capital_gain, capital_loss, hours_per_week, native_country
    )) AS duplicate_rows
FROM hw3_data_clean;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

1 rows affected.

total_rows,distinct_business_keys,duplicate_rows
45175,45170,5


In [ ]:
%%sql
-- Групи дублікатів за business key: чи збігається мітка і з яких частин датасету записи
SELECT
    age, sex, race, education, occupation, fnlwgt, hours_per_week,
    COUNT(*)                                          AS n_rows,
    STRING_AGG(income_over_50k::TEXT, ', ')            AS income_over_50k,
    STRING_AGG(CASE WHEN from_test_split THEN 'test' ELSE 'train' END, ', ') AS split
FROM hw3_data_clean
GROUP BY
    age, workclass, fnlwgt, education, education_num,
    marital_status, occupation, relationship, race, sex,
    capital_gain, capital_loss, hours_per_week, native_country
HAVING COUNT(*) > 1
ORDER BY fnlwgt;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

5 rows affected.

age,sex,race,education,occupation,fnlwgt,hours_per_week,n_rows,income_over_50k,split
39,Male,White,Bachelors,Craft-repair,138192,40,2,"true, false","train, train"
34,Female,White,Masters,Exec-managerial,169583,50,2,"true, false","test, train"
46,Male,Black,HS-grad,Transport-moving,224559,40,2,"false, true","train, test"
46,Male,White,HS-grad,Craft-repair,235646,40,2,"false, true","test, train"
41,Male,White,HS-grad,Transport-moving,242804,40,2,"false, true","test, test"


**Результат:** 45 175 рядків, 45 170 унікальних business key, **5 дублікатів** (≈ 0,01 %). Усі вони — 5 пар записів, де два рядки збігаються за всіма 14 ознаками.

**Характер дублікатів:**
**У всіх 5 парах мітки суперечать одна одній:** один рядок має `income_over_50k = true`, інший — `false`. Це очікувано: точні дублікати (з однаковою міткою) уже прибрані через `DISTINCT ON` під час завантаження, тож залишилися лише ті, що відрізняються цільовою змінною.


## 3.3. Null-rate і спеціальні маркери

Два рівні перевірки:
1. null-rate у фінальній таблиці для 8 ключових колонок;
2. маркери пропусків у сирих даних (`NULL`, порожній рядок, `?`, `unknown`, `n/a`, `none`, `null`, `nan`, `-`) для всіх 15 колонок staging;

In [ ]:
%%sql
-- Null-rate у фінальній таблиці для 8 ключових колонок
SELECT
    ROUND(COUNT(*) FILTER (WHERE age IS NULL)             * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_age,
    ROUND(COUNT(*) FILTER (WHERE workclass IS NULL)       * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_workclass,
    ROUND(COUNT(*) FILTER (WHERE education IS NULL)       * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_education,
    ROUND(COUNT(*) FILTER (WHERE occupation IS NULL)      * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_occupation,
    ROUND(COUNT(*) FILTER (WHERE sex IS NULL)             * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_sex,
    ROUND(COUNT(*) FILTER (WHERE hours_per_week IS NULL)  * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_hours_per_week,
    ROUND(COUNT(*) FILTER (WHERE native_country IS NULL)  * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_native_country,
    ROUND(COUNT(*) FILTER (WHERE income_over_50k IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_income_over_50k,
    COUNT(*) AS total_rows
FROM hw3_data_clean;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

1 rows affected.

pct_null_age,pct_null_workclass,pct_null_education,pct_null_occupation,pct_null_sex,pct_null_hours_per_week,pct_null_native_country,pct_null_income_over_50k,total_rows
0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,45175


In [ ]:
%%sql

SELECT
    col,
    COUNT(*) FILTER (WHERE val IS NULL)                         AS n_sql_null,
    COUNT(*) FILTER (WHERE TRIM(val) = '')                      AS n_empty,
    COUNT(*) FILTER (WHERE TRIM(val) = '?')                     AS n_question,
    COUNT(*) FILTER (WHERE LOWER(TRIM(val)) IN
        ('unknown', 'n/a', 'na', 'none', 'null', 'nan', '-'))   AS n_text_marker,
    COUNT(*) FILTER (WHERE val IS NULL OR TRIM(val) IN ('', '?')
        OR LOWER(TRIM(val)) IN ('unknown', 'n/a', 'na', 'none', 'null', 'nan', '-')) AS n_missing_total,
    ROUND(COUNT(*) FILTER (WHERE val IS NULL OR TRIM(val) IN ('', '?')
        OR LOWER(TRIM(val)) IN ('unknown', 'n/a', 'na', 'none', 'null', 'nan', '-'))
        * 100.0 / NULLIF(COUNT(*), 0), 2)                       AS pct_missing
FROM hw3_data_staging
CROSS JOIN LATERAL (VALUES
    ('age',            age_raw),
    ('workclass',      workclass_raw),
    ('fnlwgt',         fnlwgt_raw),
    ('education',      education_raw),
    ('education_num',  education_num_raw),
    ('marital_status', marital_status_raw),
    ('occupation',     occupation_raw),
    ('relationship',   relationship_raw),
    ('race',           race_raw),
    ('sex',            sex_raw),
    ('capital_gain',   capital_gain_raw),
    ('capital_loss',   capital_loss_raw),
    ('hours_per_week', hours_per_week_raw),
    ('native_country', native_country_raw),
    ('income',         income_raw)
) AS v(col, val)
GROUP BY col
ORDER BY n_missing_total DESC, col;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

15 rows affected.

col,n_sql_null,n_empty,n_question,n_text_marker,n_missing_total,pct_missing
occupation,966,0,1843,0,2809,5.75
workclass,963,0,1836,0,2799,5.73
native_country,274,0,583,0,857,1.75
age,0,0,0,0,0,0.00
capital_gain,0,0,0,0,0,0.00
capital_loss,0,0,0,0,0,0.00
education,0,0,0,0,0,0.00
education_num,0,0,0,0,0,0.00
fnlwgt,0,0,0,0,0,0.00
hours_per_week,0,0,0,0,0,0.00


**Результати.**

**1. Null-rate у `hw3_data_clean` — 0,00 % для всіх 8 перевірених колонок** (`age`, `workclass`, `education`, `occupation`, `sex`, `hours_per_week`, `native_country`, `income_over_50k`). Це наслідок самого дизайну таблиці: усі колонки `NOT NULL`, а рядки з пропусками відсічені під час `INSERT`. Тобто нульовий null-rate тут показує, що очищення спрацювало, а не що джерело було повним.

**2. Пропуски в сирих даних є лише в трьох колонках:** `occupation` 5,75 %, `workclass` 5,73 %, `native_country` 1,75 %. У джерелі вони закодовані двома способами — SQL `NULL` (≈ 1/3) і `'?'` (≈ 2/3), тому перевірка лише `IS NULL` занижила б частку пропусків утричі. Порожніх рядків і текстових маркерів (`unknown`, `n/a`, `none` тощо) немає в жодній із 15 колонок. Інші 12 колонок — 0 % пропусків.



# Завдання 4. Розподіли та потенційні аномалії


In [ ]:
%%sql
SELECT
    workclass,
    COUNT(*) AS n_rows,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_rows,
    -- додатково: частка доходу > 50K усередині категорії
    ROUND(AVG(income_over_50k::INT) * 100, 1) AS pct_over_50k
FROM hw3_data_clean
GROUP BY workclass
ORDER BY n_rows DESC, workclass
LIMIT 10;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

7 rows affected.

workclass,n_rows,pct_rows,pct_over_50k
Private,33262,73.63,21.8
Self-emp-not-inc,3795,8.40,27.9
Local-gov,3100,6.86,29.5
State-gov,1946,4.31,26.7
Self-emp-inc,1645,3.64,55.4
Federal-gov,1406,3.11,39.0
Without-pay,21,0.05,9.5


## 4.1. Розподіл категоріальної колонки

Колонка — `workclass` (тип зайнятості): вона описує, де і на яких умовах працює людина, і безпосередньо пов'язана з доходом.

**Інтерпретація.**

**Сильний дисбаланс.** `Private` — 73,6 % усіх рядків, у 9 разів більше за наступну категорію (`Self-emp-not-inc`, 8,4 %). Усі державні категорії разом (`Local-gov`, `State-gov`, `Federal-gov`) — 14,3 %, самозайняті (`Self-emp-*`) — 12 %. Це відповідає реальній структурі ринку праці США, тож дисбаланс — властивість популяції, а не помилка даних. Наслідок для аналізу: будь-яка статистика «по всій вибірці» фактично описує приватний сектор, а менші категорії треба дивитися окремо.

**Рідкісна категорія `Without-pay` — 21 рядок (0,05 %).**


## 4.2. Базова статистика числової колонки

Колонка — `hours_per_week` (робочі години на тиждень).

In [ ]:
%%sql
SELECT
    COUNT(hours_per_week) AS n_non_null,
    MIN(hours_per_week) AS min_value,
    MAX(hours_per_week) AS max_value,
    ROUND(AVG(hours_per_week)::NUMERIC, 2) AS avg_value,
    ROUND(STDDEV_SAMP(hours_per_week)::NUMERIC, 2) AS sd_value
FROM hw3_data_clean;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

1 rows affected.

n_non_null,min_value,max_value,avg_value,sd_value
45175,1,99,40.94,12.01


**Інтерпретація.**

| Показник | Значення | Що означає |
|---|---|---|
| `n_non_null` | 45 175 | Значення є в усіх рядках — збігається із загальною кількістю рядків, що очікувано для колонки `NOT NULL`. |
| `min_value` | 1 | Хоча б одна людина працює 1 годину на тиждень. Це можливо (епізодична робота), але нетипово — кандидат на перевірку в аналізі аномалій. |
| `max_value` | 99 | Не реальне вимірювання, а верхня межа кодування перепису («99 і більше»); її мають 123 рядки. Справжній максимум невідомий. |
| `avg_value` | 40,94 | Середній тиждень — близько 41 години, тобто практично стандартний повний робочий тиждень. |
| `sd_value` | 12,01 | Вибіркове стандартне відхилення: у середньому значення відхиляються від середнього приблизно на 12 годин. |

## 4.3. Outlier-check (доменний)

Обрано доменний варіант. У цьому датасеті кожна колонка окремо має валідні значення (діапазони вже гарантують `CHECK`), тому цікавіше шукати **неможливі комбінації** значень кількох колонок. Правила:

1. `relationship = 'Husband'`, але `sex = 'Female'`;
2. `relationship = 'Wife'`, але `sex = 'Male'`;
3. `relationship` — `Husband` або `Wife`, але `marital_status` не означає шлюбу;

In [ ]:
%%sql
-- Доменний outlier-check: рядки з логічно неможливими або неправдоподібними комбінаціями значень
WITH flagged AS (
    SELECT
        person_id, age, sex, relationship, marital_status, education,
        workclass, hours_per_week, income_over_50k,
        CASE
            WHEN relationship = 'Husband' AND sex = 'Female'
                THEN 'Husband, але sex = Female'
            WHEN relationship = 'Wife' AND sex = 'Male'
                THEN 'Wife, але sex = Male'
            WHEN relationship IN ('Husband', 'Wife')
             AND marital_status NOT IN ('Married-civ-spouse', 'Married-AF-spouse')
                THEN 'Husband/Wife, але не в шлюбі'
        END AS rule
    FROM hw3_data_clean
)
SELECT
    rule,
    COUNT(*) OVER (PARTITION BY rule) AS n_rule,
    person_id, age, sex, relationship, marital_status, education,
    workclass, hours_per_week, income_over_50k
FROM flagged
WHERE rule IS NOT NULL
ORDER BY n_rule DESC, rule, person_id;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

2 rows affected.

rule,n_rule,person_id,age,sex,relationship,marital_status,education,workclass,hours_per_week,income_over_50k
"Husband, але sex = Female",1,18528,34,Female,Husband,Married-civ-spouse,HS-grad,Private,40,False
"Wife, але sex = Male",1,12809,29,Male,Wife,Married-civ-spouse,Bachelors,Private,40,True


**Результат:** запит повернув **2 рядків з 45 175 (≈ 0,0044 %)**.

- **`Husband` + `Female` (1 рядок) і `Wife` + `Male` (1 рядок).** Тож найімовірніше це помилка кодування в одній із двох колонок (`sex` або `relationship`), але з даних не видно, яка саме неправильна.
- **`Husband`/`Wife` поза шлюбом — 0 рядків.** Перевірено, що всі записи з роллю чоловіка чи дружини мають `marital_status` `Married-civ-spouse` або `Married-AF-spouse`; ці дві колонки повністю узгоджені.

**Рішення.** Рядки залишаємо в `hw3_data_clean`: частка мізерна, а без доступу до оригінальних анкет не можна визначити, яке саме значення помилкове, тож виправлення або видалення було б догадкою. Важливо, що ці рядки пройшли всі `CHECK`-обмеження таблиці: кожне значення окремо валідне, і лише перевірка узгодженості між колонками показує проблему.

# Завдання 5. Десять DQL/EDA-запитів

### 5.1. Освіта і високий дохід

**Business-question:** Яка частка людей із доходом > 50K у кожному рівні освіти?

In [37]:
%%sql
SELECT
    education,
    education_num,
    COUNT(*)                    AS n_rows,
    SUM(income_over_50k::INT)   AS n_over_50k,
    ROUND(
        SUM(income_over_50k::INT) * 100.0 / NULLIF(COUNT(*), 0),
        1
    )                           AS pct_over_50k
FROM hw3_data_clean
GROUP BY education, education_num
ORDER BY education_num;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

16 rows affected.

education,education_num,n_rows,n_over_50k,pct_over_50k
Preschool,1,70,1,1.4
1st-4th,2,220,8,3.6
5th-6th,3,447,22,4.9
7th-8th,4,822,55,6.7
9th,5,676,38,5.6
10th,6,1223,82,6.7
11th,7,1619,89,5.5
12th,8,575,43,7.5
HS-grad,9,14770,2414,16.3
Some-college,10,9887,1988,20.1


**Interpretation:** Частка > 50K майже монотонно зростає з рівнем освіти: від 1,4 % (`Preschool`) і 3,6–7,5 % для незакінченої школи до 16,3 % у `HS-grad`, 42,0 % у `Bachelors`, 55,4 % у `Masters` і понад 73 % у `Prof-school` / `Doctorate`. Найбільший стрибок — між `Assoc-acdm` (26,4 %) і `Bachelors` (42,0 %): саме диплом бакалавра є ключовим порогом. Освіта — один із найсильніших предикторів доходу в датасеті.

### 5.2. Пропуски за типом зайнятості

**Business-question:** Які значення `workclass` у сирих даних мають найбільше пропусків у `occupation`?

In [38]:
%%sql
SELECT
    COALESCE(NULLIF(TRIM(workclass_raw), '?'), '(пропуск)') AS workclass,
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (
        WHERE occupation_raw IS NULL OR TRIM(occupation_raw) = '?'
    ) AS n_missing_occupation,
    ROUND(
        COUNT(*) FILTER (WHERE occupation_raw IS NULL OR TRIM(occupation_raw) = '?')
        * 100.0 / NULLIF(COUNT(*), 0),
        1
    ) AS pct_missing_occupation
FROM hw3_data_staging
GROUP BY COALESCE(NULLIF(TRIM(workclass_raw), '?'), '(пропуск)')
ORDER BY n_missing_occupation DESC, workclass;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

9 rows affected.

workclass,n_rows,n_missing_occupation,pct_missing_occupation
(пропуск),2799,2799,100.0
Never-worked,10,10,100.0
Federal-gov,1432,0,0.0
Local-gov,3136,0,0.0
Private,33906,0,0.0
Self-emp-inc,1695,0,0.0
Self-emp-not-inc,3862,0,0.0
State-gov,1981,0,0.0
Without-pay,21,0,0.0


**Interpretation:** Пропуски `occupation` зосереджені лише в двох групах: у всіх 2 799 рядках із пропущеним `workclass` (100 %) і в усіх 10 рядках `Never-worked` (100 %), тоді як для всіх інших типів зайнятості їх 0. Це підтверджує, що пропуски не випадкові: респондент або не відповів на весь блок питань про роботу, або професія до нього не застосовна.

### 5.3. Робочі години і дохід

**Business-question:** Як змінюється частка доходу > 50K залежно від діапазону робочих годин на тиждень?

In [ ]:
%%sql
SELECT
    CASE
        WHEN hours_per_week < 20               THEN '< 20'
        WHEN hours_per_week BETWEEN 20 AND 34  THEN '20–34'
        WHEN hours_per_week BETWEEN 35 AND 40  THEN '35–40'
        WHEN hours_per_week BETWEEN 41 AND 50  THEN '41–50'
        WHEN hours_per_week BETWEEN 51 AND 60  THEN '51–60'
        ELSE '> 60'
    END                                        AS hours_band,
    COUNT(*)                                   AS n_rows,
    ROUND(AVG(income_over_50k::INT) * 100, 1)  AS pct_over_50k
FROM hw3_data_clean
GROUP BY hours_band
ORDER BY MIN(hours_per_week);

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

6 rows affected.

hours_band,n_rows,pct_over_50k
< 20,1995,6.7
20–34,5020,6.9
35–40,24391,21.1
41–50,8569,39.8
51–60,3611,43.6
> 60,1589,36.6


**Interpretation:** Частка > 50K майже не змінюється до 35 годин (6,7–6,9 %), а потім різко зростає: 21,1 % для стандартного тижня 35–40 годин, 39,8 % для 41–50 і максимум 43,6 % для 51–60. Понад 60 годин частка знову знижується до 36,6 %. Тож зв'язок годин із доходом нелінійний.

### 5.4. Рідкісні країни походження

**Business-question:** Які 10 країн походження трапляються в даних найрідше?

In [ ]:
%%sql
SELECT
    native_country,
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 3) AS pct_rows
FROM hw3_data_clean
GROUP BY native_country
ORDER BY n_rows ASC, native_country
LIMIT 10;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

10 rows affected.

native_country,n_rows,pct_rows
Holand-Netherlands,1,0.002
Hungary,18,0.040
Honduras,19,0.042
Scotland,20,0.044
Laos,21,0.046
Outlying-US(Guam-USVI-etc),22,0.049
Yugoslavia,23,0.051
Cambodia,26,0.058
Trinadad&Tobago,26,0.058
Hong,28,0.062


**Interpretation:** Найрідша країна — `Holand-Netherlands`, лише 1 запис; далі йдуть `Hungary` (18), `Honduras` (19), `Scotland` (20) і ще 6 країн з 21–28 записами, кожна менше 0,07 % даних. За такої кількості будь-які статистики по країнах (наприклад, частка > 50K) ненадійні, а категорія з 1 записом гарантовано потрапить лише в train або лише в test. Для моделі рідкісні країни доцільно згрупувати в `Other` або за регіонами.

### 5.5. Сервісні професії

**Business-question:** Яка частка високого доходу в професіях сфери обслуговування (у назві є `serv`)?

In [39]:
%%sql
SELECT
    occupation,
    COUNT(*)                                   AS n_rows,
    ROUND(AVG(hours_per_week), 1)              AS avg_hours,
    ROUND(AVG(income_over_50k::INT) * 100, 1)  AS pct_over_50k
FROM hw3_data_clean
WHERE occupation ILIKE '%serv%'
GROUP BY occupation
ORDER BY pct_over_50k DESC, occupation;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

3 rows affected.

occupation,n_rows,avg_hours,pct_over_50k
Protective-serv,975,42.8,31.5
Other-service,4805,34.7,4.1
Priv-house-serv,230,32.8,1.3


**Interpretation:** `ILIKE '%serv%'` знайшов три професії, які сильно відрізняються між собою. `Protective-serv` (поліція, охорона, пожежні) має 31,5 % > 50K і 42,8 години на тиждень — вище за середнє по вибірці, тоді як `Other-service` (4,1 %) і `Priv-house-serv` (1,3 %, домашня прислуга) — серед найнижчих і з коротшим тижнем (~33–35 годин). Тож спільне слово в назві не означає однорідної групи: об'єднувати ці професії в одну категорію «сервіс» для аналізу чи моделі було б помилкою.

### 5.6. Високий дохід без вищої освіти

**Business-question:** Скільки працездатних (25–64 роки) людей без вищої освіти з доходом > 50K мають капітальний прибуток або працюють понад 50 годин на тиждень?

In [40]:
%%sql
SELECT
    COUNT(*) AS n_capital_or_overtime,
    (
        SELECT COUNT(*)
        FROM hw3_data_clean
        WHERE income_over_50k
          AND NOT (education IN ('Bachelors', 'Masters', 'Prof-school', 'Doctorate'))
          AND age BETWEEN 25 AND 64
    ) AS n_all_no_degree_over_50k
FROM hw3_data_clean
WHERE income_over_50k
  AND NOT (education IN ('Bachelors', 'Masters', 'Prof-school', 'Doctorate'))
  AND age BETWEEN 25 AND 64
  AND (capital_gain > 0 OR hours_per_week > 50);

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

1 rows affected.

n_capital_or_overtime,n_all_no_degree_over_50k
1773,5409


**Interpretation:** Серед 5 409 людей 25–64 років без вищої освіти з доходом > 50K у 1 773 (≈ 32,8 %) є капітальний прибуток або тиждень понад 50 годин. Тобто приблизно третина таких людей компенсує відсутність диплома додатковим джерелом доходу або тривалим робочим часом. Решта ≈ 67 % досягають > 50K без цих факторів, тож їхній дохід треба пояснювати іншими ознаками (професія, стаж, сімейний стан) — освіта важлива, але не єдиний шлях до високого доходу.

### 5.7. Відповідність `education` і `education_num`

**Business-question:** Чи кожна текстова категорія `education` має рівно один числовий код `education_num`?


In [ ]:
%%sql
SELECT DISTINCT
    education_num,
    education
FROM hw3_data_clean
ORDER BY education_num, education;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

16 rows affected.

education_num,education
1,Preschool
2,1st-4th
3,5th-6th
4,7th-8th
5,9th
6,10th
7,11th
8,12th
9,HS-grad
10,Some-college


**Interpretation:** Запит повернув 16 унікальних пар, і кожен код від 1 (`Preschool`) до 16 (`Doctorate`) трапляється рівно один раз — відповідність `education` ↔ `education_num` є строго один-до-одного, суперечливих комбінацій немає. Отже, дві колонки несуть одну й ту саму інформацію: `education_num` — порядкове кодування `education`. Для моделі достатньо однієї з них, інакше ознаки дублюватимуться; зручніше лишити `education_num`, бо вона вже зберігає порядок рівнів.

### 5.8. Що змінює правило очищення `workclass`

**Business-question:** Скільки сирих значень `workclass` змінює правило `NULLIF(TRIM(workclass_raw), '?')` і чи помітить ці зміни звичайне порівняння `<>`?

In [ ]:
%%sql
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE workclass_raw IS NULL) AS n_raw_null,
    COUNT(*) FILTER (
        WHERE NULLIF(TRIM(workclass_raw), '?') IS DISTINCT FROM workclass_raw
    ) AS n_changed_is_distinct_from,
    COUNT(*) FILTER (
        WHERE NULLIF(TRIM(workclass_raw), '?') <> workclass_raw
    ) AS n_changed_not_equal
FROM hw3_data_staging;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

1 rows affected.

n_rows,n_raw_null,n_changed_is_distinct_from,n_changed_not_equal
48842,963,1836,0


**Interpretation:** З 48 842 сирих значень правило змінює 1 836 — це рівно кількість маркерів `'?'`, що перетворюються на `NULL`; 963 сирі `NULL` залишаються `NULL` і слушно не вважаються зміненими. Звичайне `<>` показало **0** змін: порівняння `NULL <> '?'` повертає `NULL`, а не `TRUE`, і такий рядок не проходить фільтр. Тобто саме ті значення, які очищення змінює, порівняння `<>` пропускає всі. Для перевірок «до/після» у колонках, де може бути `NULL`, потрібне саме `IS DISTINCT FROM`.

### 5.9. Професії з найвищою часткою високого доходу

**Business-question:** Які п'ять професій (щонайменше 100 записів) мають найбільшу частку доходу > 50K?

In [ ]:
%%sql
SELECT
    occupation,
    COUNT(*)                                   AS n_rows,
    ROUND(AVG(income_over_50k::INT) * 100, 1)  AS pct_over_50k
FROM hw3_data_clean
GROUP BY occupation
HAVING COUNT(*) >= 100
ORDER BY pct_over_50k DESC, occupation
LIMIT 5;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

5 rows affected.

occupation,n_rows,pct_over_50k
Exec-managerial,5980,47.9
Prof-specialty,6001,45.0
Protective-serv,975,31.5
Tech-support,1419,29.0
Sales,5405,26.9


**Interpretation:** Лідери — `Exec-managerial` (47,9 %) і `Prof-specialty` (45,0 %): у цих професіях майже кожен другий заробляє понад 50K, і це найбільші групи (≈ 6 тис. записів кожна), тож оцінки надійні. Далі з помітним розривом йдуть `Protective-serv` (31,5 %), `Tech-support` (29,0 %) і `Sales` (26,9 %) — лише трохи вище за середнє по вибірці (24,8 %).

### 5.10. Стать, шлюб і дохід

**Business-question:** Як відрізняється частка доходу > 50K у чоловіків і жінок залежно від того, чи вони перебувають у шлюбі?

In [ ]:
%%sql
SELECT
    sex,
    CASE
        WHEN marital_status IN ('Married-civ-spouse', 'Married-AF-spouse')
            THEN 'в шлюбі'
        ELSE 'не в шлюбі'
    END                                        AS marital_group,
    COUNT(*)                                   AS n_rows,
    ROUND(AVG(hours_per_week), 1)              AS avg_hours,
    ROUND(AVG(income_over_50k::INT) * 100, 1)  AS pct_over_50k
FROM hw3_data_clean
GROUP BY sex, marital_group
ORDER BY sex, marital_group;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw3_pg'

4 rows affected.

sex,marital_group,n_rows,avg_hours,pct_over_50k
Female,в шлюбі,2234,37.3,46.6
Female,не в шлюбі,12446,36.9,5.1
Male,в шлюбі,18840,44.6,45.3
Male,не в шлюбі,11655,40.1,8.6


**Interpretation:** Відсоток доходу > 50K майже однаковий у чоловіків і жінок у шлюбі (45,3 % і 46,6 %), але поза шлюбом падає до 8,6 % у чоловіків і 5,1 % у жінок. Тож сімейний стан розділяє дохід сильніше за стать, частково через вік (неодружені в середньому молодші).

# Завдання 6. Reflection



**1. Проблеми якості.** Пропуски в `workclass`, `occupation`, `native_country` закодовані двома способами (`NULL` і `'?'`) і не випадкові: вони пов'язані з нижчим доходом, а для `Never-worked` `'?'` означає «не застосовно». Мітка `income` мала чотири написання, бо крапка позначала тестовий файл. Знайдено точні дублікати та пари з однаковим профілем, але різною міткою. Числові колонки обрізані зверху (`age = 90`, `hours_per_week = 99`, `capital_gain = 99999`), є неоднозначні категорії (`South`, `Hong`, `Other`), описки (`Holand-Netherlands`) і поодинокі суперечності між колонками (`Husband` + `Female`).

**2. Features.** Використав би `age`, `workclass`, `education_num`, `marital_status`, `occupation`, `relationship`, `hours_per_week` і `native_country` (рідкісні країни — в `Other`). Не використав би `education` (дублює `education_num`), `fnlwgt` (вага вибірки, а не властивість людини; краще як `sample_weight`), `person_id` і `from_test_split`. `sex` і `race` залишив би лише для аудиту упередженості моделі, а не як ознаки.

**3. Leakage.** Ризик — `capital_gain` і `capital_loss`: це річні суми, які є частиною того самого доходу, що й target, і стають відомими лише після завершення року. Якщо прогноз робиться заздалегідь, їх треба виключити.

**4. Пропозиції data engineer.** Єдине кодування пропусків (`NULL`), окрема колонка для train/test замість крапки в мітці, ідентифікатор респондента, прапорці для top-coded значень, ISO-коди країн і автоматичні перевірки узгодженості та дублікатів перед публікацією файлу.

**5. ML-задача** — бінарна класифікація (`income_over_50k`) з дисбалансом ≈ 25/75, тому потрібні стратифікований поділ і метрики на кшталт F1, а не лише accuracy.